# TSE
# WEB MINING - 2025

Instructors : Lynda Tamine and Jesús Lovón

---




### Attention❗ About PW:
🚨 *Code questions*: Fill in the missing code in the corresponding sections.

🚨 *Open questions*: Write your textual answer as a comment in the corresponding cells.

🚨 *Keep your outputs* for the cells where you write the code.

---

# PW 2: Opinion Mining


In this session, we will explore the application of opinion mining in text data. Opinion mining is a natural language processing (NLP) task that involves determining the emotional tone, **sentiment** or **opinion** expressed in a piece of text.

We will focus on sentence-level and document-level opinion using a combination of sparse and dense representations, including: **TF-IDF**, **Word-Embeddings**, and **Transformers** models, as previously explored (PW1).
Before starting working with the Transformers-based models, **please select T4 GPU in the menu: Runtime/Change runtime type and re-run these 2 cells**. Save your work before changing the runtime type.

As an example, consider the following document:

```txt
Guessing Market's Next Move: This Week's Economic Data May Give Hint Guessing Next Market Move. When the stock market
opens this morning, investors will begin to get a better idea of whether last week’s record decline was largely a
two-day aberration or the beginning of a prolonged slide in stock market prices.

The market’s course in coming days should also help clarify the extent to which the abrupt plunge last week was rooted
in the market’s own complicated workings — or, by contrast, reflected a deepening apprehension over the economy.

i Street a clearer picture of where the economy stands. Investors will be able to make fresh estimates of inflation,
the direction of interest rates •and the vigor of the economy. Concern Over Rates Financial experts have said it was a
growing concern about interest rates that touched off the battering the market took last week, particularly last
 Thursday and Friday, when the Dow Jones industrials fell almost 121 points in two days, closing the week at 1,758.72.
 The Street was deciding that interest rates would fall no further, which boded poorly for the economy.

 Some Wall Street people contend it {Will take a major development, such as a notable drop in interest rates abroad,
 to cheer up the market The financial markets believe that the Federal Reserve Board is unwilling to lower American
 interest rates any further, unless rates in West Germany and Japan come down as well.

 Indeed, many Wall Street professionals are betting that stock prices will continue to fall in coming days.

```

We can infer, from the overall document (document-level) a **negative** opinion about the stock market


Thsi example belong to the ``Sentiment Economy News Dataset``, which contains real-world financial news articles annotated with opinion labels, and which we will use in this PW.

# Objectives of the PW

Through this PW, you will:

1. Compute **sentence**-level and **document**-level representations using different appoaches.
2. Compute the opinions (**positive** and **negative**) expressed in these texts.
3. Visualize and interpret sentiment distribution.
4. Compare the effectiveness of different representations in classification tasks.

By the end of this PW, you will gain experience with various opinion mining techniques and understand the strengths and limitations of each technique.


# Before start:
Note that some code blocks are given as a guide to help you find the right answer, others to download the information or make the necessary installations/configurations.

In [ ]:
# Requierd installations
!pip install gensim
!pip install datasets
!pip install scikit-learn matplotlib
!pip install transformers torch
!pip install nltk

# I. Dataset understanding

We begin by analyzing the dataset to understand its structure and the information it provides.

The dataset consists of three subsets: train, dev and test. In this session, we will first focus on the ``test`` subset.



1. Load the dataset `MoritzLaurer/sentiment_economy_news` using the [`load_dataset`](https://huggingface.co/docs/datasets/loading) function from the Datasets library.
2.  Filter the dataset to retain only the ``test`` subset.
3. List all the columns in the dataset to get an overview of the available data.
4. Compute the average length (and standard deviation) of documents (# words), similarly to the previous PW.
5.  Print three sample sentences to inspect the dataset content.


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.manifold import TSNE
import gensim.downloader as api
import datasets
from nltk.tokenize import word_tokenize
import numpy as np

#1
ds = datasets.load_dataset('MoritzLaurer/sentiment_economy_news')
# 2
my_dataset = ds['test']
# 3
print(my_dataset.column_names)

#4
def compute_stats(list_of_sentences):
  len_list_sentences = [len(s.split()) for s in list_of_sentences]
  average_length = np.mean(len_list_sentences)
  std_dev_length = np.std(len_list_sentences)
  print(f"Avg length: {average_length}. Std Dev length: {std_dev_length}")

all_docs = my_dataset['text']
print(compute_stats(all_docs))
# 5. Print 3 sentences
my_dataset['text'][:3]

In [ ]:
my_sentence = my_dataset['text'][101]

# II. Opinion Mining (sentence-level)

We will perform opinion mining at the **sentence level** using the following pipeline:


- a) Tokenize paragraphs: split the input paragraph into sentences using a sentence tokenizer.
- b) Assign word-level opinion scores: for each sentence, compute opinion scores (positive and negative) for each word.
- c) Aggregate scores: combine the word-level opinion scores to determine the overall opinion for each sentence.
- d) Store the results: maintain a list of the computed opinions for all the sentences.
- e) Visualize representations: plot the sentence-level sentiment representations and analyze the resulting space.


Note that to compute opinion scores, we need additional information beyond traditional methods such as TF-IDF and word embeddings. For this purpose, we will use a human-annotated sentiment lexicon, [SentiWordNet](https://github.com/aesuli/SentiWordNet), to assign positive and negative sentiment scores to words.

 ## a) Using word counts

 We compute sentence-level opinion using a TF-IDF based approach (refer to the course materials for more details). This combines Term Frequency (TF) with opinion scores obtained from SentiWodNet.

 Refering to our main pipeline, we will compute the step ``b) and c)`` using the following objectives:

 **Step b) Assign Word-Level Opinion Scores**


 For each word w in a sentence, we follow these steps:

 1. Compute term frequency, ``tf(w)``, for the word ``w``.
 2. Use SentiWordnet to retrieve its synset (sense defintion) using the ``senti_synset`` function. If multiple synsets are available, we simply consider the first one (the most common).
 3. Compute the word's positive sentiment score using the [``pos_score()``](https://www.nltk.org/api/nltk.corpus.reader.sentiwordnet.html) function. (similarly, we compute the negative sentiment score using ``neg_score()``)

**Step c) Aggregate Scores**

To aggregate word-level sentiment into a sentence-level score:

 1. Positive Opinion Score: we aggregate these scores using the following equation:

 $ S^{+} = Σ_{w ∈ sentence} ( tf(w) * pos\_score(w) )$

 2. Negative Opinion Score: Similarly, we compute the negative score using the ``neg_score()`` function, and we obtain:

 $ S^{-} = Σ_{w ∈ sentence} ( tf(w) * neg\_score(w) )$

 3. Determine opinion:
  - A sentence is classified as positive if $S^{+} \gt S^{-}$
  - A sentence is classified as negative if $S^{-} \gt S^{+}$


In [ ]:
# a) We tokenize the paragraph into sentence using a sentence tokenizer.
nltk.download('punkt_tab')
def tokenize_paragraph(paragraph):
    return sent_tokenize(paragraph)

In [ ]:
# b) For each sentence, we obtain a sentiment score (positive and negative) for each word
# c) We aggregate the scores of each word to obtain a sentence-level sentiment

from nltk.corpus import sentiwordnet as swn
nltk.download('wordnet')
nltk.download('sentiwordnet')

def tf(list_words):
  """
  Input: A list of words
  Ouptut: A dictionary with {word: frequency}
  """
  result = {}
  for word in list_words:
      if word in result.keys():
          result[word] += 1
      else:
          result[word] = 1
  return result

def get_sentiment_sentence(sentence):
  tokens = nltk.word_tokenize(sentence)
  tfs = tf(tokens)

  pos_score = 0
  neg_score = 0
  # Iterate each word and compute the sentiment score from SentiWordNet (swn)
  for word in tokens:
      synsets = list(nltk.corpus.wordnet.synsets(word))
      if synsets:
          # Take only 1 synset, the first one, most common
          swn_synset = swn.senti_synset(synsets[0].name())
          pos_score += tfs[word]*swn_synset.pos_score()
          neg_score += tfs[word]*swn_synset.neg_score()

  return 'positive' if pos_score>neg_score else 'negative'



In [ ]:
# Compute sentiments
sentences_from_doc101 = sent_tokenize(my_sentence)
sentences_opinions = [get_sentiment_sentence(s) for s in sentences_from_doc101]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

####  Question ✍
Compute the opinion of each sentece from the document 151 from our dataset.

Print each sentence and its corresponding opinion

In [ ]:
#### YOUR CODE HERE


In [ ]:
# Solution
sentences_from_doc151 = sent_tokenize(my_dataset['text'][151])
sentences_opinions = [get_sentiment_sentence(s) for s in sentences_from_doc151]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

At this point, we can not compute or plot representation because we are working with only frequencies-based approaches. We will solve this problem using the word embeddings approach.

##  b) Using word-embeddings (dense vector semantics)

We use word-embeddings to perform opinion mining, following a similar pipeline as before but with modifications to steps ``b) and c)``.


**Step b) Compute Positive and Negative Barycenters**

Instead of computing sentiment scores for each word, we calculate a ``positive`` ($b^{+}$) and a ``negative`` ($b^{-}$) barycenters using the SentiWordnet lexicon:

1. Retrieve all available synsets from SentiWordNet using the ``all_senti_synsets`` function
2. For each synset, compute its positive and negative scores to classify it as either positive or negative.
3. Retrieve the word embedding for the synset's associated word.
4. Maintain two separate lists of word embeddings: one for positive words and one for negative words.
5.  Compute the positive and negative [barycenters](https://fr.wikipedia.org/wiki/Barycentre) ($b^{+}$, $b^{-}$) as the weighted averages (sentiment score) of their respective word embeddings.

**Step c) Aggregate Scores**

To determine the opinion of a sentence:

1. Compute a vector semantic $v(sentence)$ for the sentence as in the previous PW.
2. Compute two distances, a positive distance, $d^{+}_{sentence} = dist(v(sentence), b^{+})$, and a negative distance, $d^{-}_{sentence} = dist(v(sentence), b^{-})$.
3. Then, the sentence is positive if $d^{+} \lt d^{-}$, or negative otherwise.

This method uses the barycenters to establish a reference for positive and negative sentiments, allowing us to assess the sentence's sentiment based on its proximity to these references.


In [ ]:

###
def get_word_embeddings():
  # Download and load a pre-trained Word2Vec model
  nltk.download('punkt_tab')
  model = api.load("word2vec-google-news-300")
  return model

def get_w2v_vector(word, model):
  if word in model:
    return model[word]
  else:
    return None

w2v_model = get_word_embeddings()

In [ ]:
# a) We tokenize the paragraph into sentence using a sentence tokenizer.
def tokenize_paragraph(paragraph):
    return sent_tokenize(paragraph)

In [ ]:
# Get baricenter positive, negative
# Collect positive and negative words with scores from SentiWordNet
from nltk.tokenize import word_tokenize
nltk.download("stopwords")
from nltk.corpus import stopwords

positive_embeddings = []
negative_embeddings = []


all_synsets = list(swn.all_senti_synsets()) # Get all synset from sentiwordnet


for synset in all_synsets:
    # Extract words, positivity, and negativity scores
    words = synset.synset.lemma_names()
    pos_score = synset.pos_score()
    neg_score = synset.neg_score()

    for word in words:
        embedding = get_w2v_vector(word, w2v_model)
        if embedding is not None:
            if pos_score > neg_score:
                positive_embeddings.append((embedding, pos_score))
            elif neg_score > pos_score:
                negative_embeddings.append((embedding, neg_score))

# Compute weighted barycenters for positive and negative sentiments
def compute_barycenter(embeddings_with_scores):
    if not embeddings_with_scores:
        return None

    embeddings, scores = zip(*embeddings_with_scores)
    scores = np.array(scores)
    embeddings = np.array(embeddings)

    # Normalize scores to sum to 1 (weights)
    weights = scores / scores.sum()
    barycenter = np.dot(weights, embeddings)
    return barycenter

positive_barycenter = compute_barycenter(positive_embeddings)
negative_barycenter = compute_barycenter(negative_embeddings)



# Function to compute sentence vector
def compute_sentence_vector(sentence):

    stop_words = set(stopwords.words("english"))
    tokens = word_tokenize(sentence.lower())
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]

    embeddings = [get_w2v_vector(word, w2v_model) for word in tokens if get_w2v_vector(word, w2v_model) is not None]
    if not embeddings:
        return None

    # Compute average embedding for the sentence
    sentence_vector = np.mean(embeddings, axis=0)
    return sentence_vector

# Determine sentiment based on barycenters
def get_sentiment_sentence_w2v(sentence, positive_barycenter, negative_barycenter):
    sentence_vector = compute_sentence_vector(sentence)
    if sentence_vector is None:
        #"Neutral or no sentiment detected"
        return None

    # Compute distances to barycenters
    distance_to_positive = np.linalg.norm(sentence_vector - positive_barycenter)
    distance_to_negative = np.linalg.norm(sentence_vector - negative_barycenter)

    if distance_to_positive < distance_to_negative:
        return "positive"
    else:
        return "negative"

In [ ]:
# Compute sentiments
sentences_from_doc101 = sent_tokenize(my_sentence)
sentences_opinions = [get_sentiment_sentence_w2v(s, positive_barycenter, negative_barycenter) for s in sentences_from_doc101]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

####  Question ✍
Similarly than before, compute the opinion of each sentece from the document 151 from our dataset using word embeddings.

Print each sentence and its corresponding opinion

In [ ]:
#### YOUR CODE HERE

In [ ]:
# Solution
sentences_from_doc151 = sent_tokenize(my_dataset['text'][151])
sentences_opinions = [get_sentiment_sentence_w2v(s, positive_barycenter, negative_barycenter) for s in sentences_from_doc151]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

In [ ]:
### # e) We plot their representations and analyze the space representation.
import matplotlib.pyplot as plt
sentence_vectors = [compute_sentence_vector(sentence) for sentence in sentences_from_doc101]
sentence_vectors = [vec for vec in sentence_vectors if vec is not None]  # Remove None entries

# Add barycenters to the sentence vectors
all_vectors = np.array(sentence_vectors + [positive_barycenter, negative_barycenter])

# Perform t-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(all_vectors)

# Separate reduced embeddings
sentence_reduced = reduced_vectors[:len(sentence_vectors)]
positive_reduced = reduced_vectors[len(sentence_vectors)]
negative_reduced = reduced_vectors[len(sentence_vectors) + 1]

# Plot the results
plt.figure(figsize=(10, 6))

# Plot sentence embeddings
for s, l in zip(sentence_reduced,sentences_opinions):
  if l =='positive':
    plt.scatter(s[0], s[1], c="red",  alpha=0.6)
  else:
    plt.scatter(s[0], s[1], c="green", alpha=0.6)

# Plot barycenters
plt.scatter(positive_reduced[0], positive_reduced[1], c="green", label="Positive Barycenter", s=100)
plt.scatter(negative_reduced[0], negative_reduced[1], c="red", label="Negative Barycenter", s=100)

# Annotate sentences
for i, sentence in enumerate(sentences_from_doc101):
    if i < len(sentence_reduced):
        plt.annotate(f"Sentence {i+1}", (sentence_reduced[i, 0], sentence_reduced[i, 1]))

plt.title("t-SNE Visualization of Sentence Embeddings and Barycenters")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.legend()
plt.grid()
plt.show()

## c) [optional] Opinion mining with transformers (BERT)

We use a Transformers-based model for opinion mining. Unlike the previous methods, this approach is more direct, as the model itself provides sentiment predictions based on pre-trained weights. Specifically, we use the model ``distilbert-base-uncased-finetuned-sst-2-english``, which is fine-tuned for sentiment analysis.

The sentiment is determined from the logits (raw scores) of the model's final layer in the function ``determine_sentiment_transformers()``. For further details, you can read the documentation of [the AutoModelForSequenceClassification](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification), where you can find how to obtain the ``scores`` from the output of the model (logits).

Then, we compute probabilities based on the logits values, using the ``softmax`` function available in the ``torch`` library.

Finally, you can identify the correct label based on these probabilities and assign it to ``positive`` or ``negative``.

In [ ]:
# a) We tokenize the paragraph into sentence using a sentence tokenizer.
nltk.download('punkt_tab')
def tokenize_paragraph(paragraph):
    return sent_tokenize(paragraph)

In [ ]:

import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
# Load pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

# Define sentiment analysis function
def determine_sentiment_transformers(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    scores = torch.nn.functional.softmax(outputs.logits, dim=1)
    label_id = torch.argmax(scores).item()
    label = "positive" if label_id == 1 else "negative"
    return label

In [ ]:
# Compute sentiments
sentences_from_doc101 = sent_tokenize(my_sentence)
sentences_opinions = [determine_sentiment_transformers(s) for s in sentences_from_doc101]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

####  Question ✍
Similarly than before, compute the opinion of each sentece from the document 151 from our dataset using word embeddings.

Print each sentence and its corresponding opinion

In [ ]:
### YOUR CODE HERE

In [ ]:
# Solution
sentences_from_doc151 = sent_tokenize(my_dataset['text'][151])
sentences_opinions2 = [determine_sentiment_transformers(s) for s in sentences_from_doc151]


for a_sentence, opinion in zip(sentences_from_doc101, sentences_opinions):
  print("Sentence:", a_sentence)
  print("Opinion:", opinion)
  print("---")

In [ ]:
### # e) We plot their representations and analyze the space representation.

# Find vector representation of each sentence and plot the representations
# Use different color for positive and negative sentences
# Note that there are not barycenters in this case

!pip install -U sentence-transformers
#!pip install -U accelerate

from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

model_name = "distilbert-base-uncased-finetuned-sst-2-english"  # You can use any model available
enc_model = SentenceTransformer(model_name)

sentence_vectors = enc_model.encode(sentences_from_doc101)

# Perform t-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=1)
reduced_vectors = tsne.fit_transform(sentence_vectors)

# Plot the results
plt.figure(figsize=(10, 6))


# Plot sentence embeddings
for s, l in zip(reduced_vectors,sentences_opinions):
  if l =='positive':
    plt.scatter(s[0], s[1], c="red",  alpha=0.6)
  else:
    plt.scatter(s[0], s[1], c="green", alpha=0.6)

# Annotate sentences
for i, sentence in enumerate(reduced_vectors):
    if i < len(reduced_vectors):
        plt.annotate(f"Sentence {i+1}", (reduced_vectors[i, 0], reduced_vectors[i, 1]))

plt.title("t-SNE Visualization of Sentence Embeddings and Barycenters")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.legend()
plt.grid()
plt.show()

####  Question ✍

1. Make two tables to regroup the three different results (using word count, word embeddings and Transformers), one table for each entry (101, 151).

2. Do the models agree on their predictions? Why?

3. You, as a human annotator, will assign a 'correct' opinion to each sentence. Make a ranking to determine the best model to predict opinion, based on these examples.

# III. Evaluation of Opinion Mining (document-level)


In this section, we compute our opinion at the document level, as shown in our first example.

Unlike the previous experiments where we tokenized paragraphs into sentences, here we process the entire document as a single unit. This allows us to analyze the overall opining of a text and evaluate how well the models perform on larger contexts.

Repeat the experimentations similarly than before. Instead of tokenization the paragraph into sentences and then into words, we will use the complete document and tokenize it into words (in particular, for opinion mining methods based on **word counts** and **word embeddings**).

### Evaluation Metrics:
To measure the performance of each model, compute the following metrics:

- Accuracy: Proportion of correctly predicted labels out of all predictions.
- Recall: Measure of the model's ability to identify true positives.
- F1-Score: Harmonic mean of precision and recall, representing a balanced evaluation metric.
- Confusion Matrix: Matrix summarizing true/false positives and negatives for visual analysis.


In the dataset, you have the ``labels`` column, which provide the right sentiment at the ``document-level``. This time, instead of computing only one entry, we will compute the complete dataset.

We implement and ``evaluate model`` function that will compute the previous metrics based on model predictions and labels.

In [ ]:
# Evaluation Metrics implemenation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


def evaluate_model(predictions, labels):
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", pos_label="positive")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-Score:", f1)
    print("\Confusion Matrix:\n", confusion_matrix(labels, predictions))

In [ ]:
# We get the labels and all the sentences from the dataset
labels, my_minicorpus = my_dataset['labels'], my_dataset['text']

## a) Using word count

####  Question ✍
Using the previous implemented functions, compute the opinion of all sentences.

In [ ]:
#### YOUR CODE HERE
tfidf_predictions = ...

In [ ]:
# Solution
tfidf_predictions = [get_sentiment_sentence(sent) for sent in my_minicorpus ]

In [ ]:
# We compute its performance
evaluate_model(tfidf_predictions, labels)

## Using word embeddings

####  Question ✍
Using the previous implemented functions, compute the opinion of all sentences.

In [ ]:
#### YOUR CODE HERE
w2v_predictions = ...

In [ ]:
# Solution
w2v_predictions = [get_sentiment_sentence_w2v(sent,positive_barycenter, negative_barycenter) for sent in my_minicorpus ]

In [ ]:
evaluate_model(w2v_predictions, labels)

## [Optional] Using transformers

####  Question ✍
Using the previous implemented functions, compute the opinion of all sentences.

In [ ]:
#### YOUR CODE HERE


In [ ]:
# Solution
bert_predictions = [determine_sentiment_transformers(sent) for sent in tqdm(my_minicorpus) ]

In [ ]:
evaluate_model(bert_predictions, labels)

####  Question ✍

Analyze and provide an interpretation for the different performances between models.

In [ ]:
# Votre code ici